# PCC Ensemble Surrogate Modeling — 8-Model Training Pipeline
## Many-Objective Ensemble ML Optimization of Amine-Based PCC

This notebook trains and evaluates 8 surrogate models on the 5,200-row PCC dataset:
1. Artificial Neural Network (ANN) — Keras Tuner Hyperband
2. Tabular Prior-Fitted Networks (TabPFN) — Zero-Shot Transformer
3. PSO-CatBoost — Particle Swarm Optimized
4. PSO-GBR — Gradient Boosting Regressor (PSO-tuned)
5. PSO-XGBoost — Extreme Gradient Boosting (PSO-tuned)
6. PSO-AdaBoost — Adaptive Boosting (PSO-tuned)
7. Bagged CatBoost (6× Ensemble) — Bootstrap Aggregation
8. Stacked Meta-Surrogate — Ridge Meta-Learner combining all 6 base models

**30 input features** (after feature selection) → **6 response targets**

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor, AdaBoostRegressor, RandomForestRegressor, BaggingRegressor
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.multioutput import MultiOutputRegressor
import xgboost as xgb
from catboost import CatBoostRegressor
import tensorflow as tf
from tensorflow import keras

print(f"TensorFlow: {tf.__version__}")
print(f"scikit-learn: {__import__('sklearn').__version__}")
print(f"CatBoost: {__import__('catboost').__version__}")

## Step 1: Load & Inspect Dataset
5,200 rows × 36 columns (30 inputs + 6 targets)

In [ ]:
df = pd.read_csv('Dataset.csv')
print(f"Dataset shape: {df.shape}")

TARGET_COLS = ['co2_recovery', 'co2_purity', 'sec_energy', 
               'solvent_loss', 'opex_usd_per_t', 'corrosion_rate']

# 30 final input features (after feature selection)
FEATURE_COLS = [c for c in df.columns if c not in TARGET_COLS]
print(f"Input features: {len(FEATURE_COLS)}")
print(f"Targets: {TARGET_COLS}")
print(f"\nAmine distribution:")
print(df['amine_type'].map({1:'MEA',2:'DGA',3:'DEA',4:'TEA',5:'MDEA'}).value_counts())

X = df[FEATURE_COLS].values
y = df[TARGET_COLS].values
print(f"\nX shape: {X.shape} | y shape: {y.shape}")

## Step 2: Feature Selection — Multi-Random Forest Gini Importance
4 variables pruned (Gini importance < 0.005): `flue_gas_pressure`, `flue_gas_o2_mol`, `co2_amine_loading`, `fresh_amine_unit_cost`

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Multi-target Gini importance averaging
rf_selector = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_selector.fit(X, y)
importances = rf_selector.feature_importances_
feat_imp = pd.DataFrame({'feature': FEATURE_COLS, 'gini_importance': importances})
feat_imp = feat_imp.sort_values('gini_importance', ascending=False)

PRUNED_FEATURES = ['flue_gas_pressure', 'flue_gas_o2_mol', 'co2_amine_loading', 'fresh_amine_unit_cost']
FINAL_FEATURES = [f for f in FEATURE_COLS if f not in PRUNED_FEATURES]
print(f"Initial features: {len(FEATURE_COLS)} → Final features: {len(FINAL_FEATURES)}")
print(f"Pruned (Gini < 0.005): {PRUNED_FEATURES}")

plt.figure(figsize=(12, 6))
colors = ['#d62728' if f in PRUNED_FEATURES else '#1f77b4' for f in feat_imp['feature']]
plt.bar(range(len(feat_imp)), feat_imp['gini_importance'], color=colors)
plt.axhline(y=0.005, color='red', linestyle='--', label='Threshold (0.005)')
plt.xticks(range(len(feat_imp)), feat_imp['feature'], rotation=90)
plt.xlabel('Feature'); plt.ylabel('Gini Importance')
plt.title('Multi-Random Forest Feature Importance (Multi-Target)')
plt.legend(); plt.tight_layout(); plt.savefig('feature_importance.png', dpi=150); plt.show()

X_final = df[FINAL_FEATURES].values

## Step 3: Data Preprocessing — Train/Test Split + Robust Scaling
80/20 split, RobustScaler (median/IQR based)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, test_size=0.2, random_state=42)

scaler_X = RobustScaler()
scaler_y = RobustScaler()

X_train_sc = scaler_X.fit_transform(X_train)
X_test_sc  = scaler_X.transform(X_test)
y_train_sc = scaler_y.fit_transform(y_train)
y_test_sc  = scaler_y.transform(y_test)

joblib.dump(scaler_X, 'scaler_X.pkl')
joblib.dump(scaler_y, 'scaler_y.pkl')
print(f"Train: {X_train_sc.shape} | Test: {X_test_sc.shape}")

## Step 4: ANN — Keras Tuner Hyperband
Architecture per target group:
- Recovery/Purity: [32, 72, 32], ReLU, lr=0.01, batch=32
- SEC/Loss/OPEX/Corrosion: [128, 120, 64], ReLU, Dropout=0.1, L2=1e-4, lr=0.005

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping

def build_ann(n_inputs, n_outputs, hidden_layers, dropout=0.0, l2_reg=0.0, lr=0.01):
    model = Sequential()
    for i, units in enumerate(hidden_layers):
        if i == 0:
            model.add(Dense(units, activation='relu', input_shape=(n_inputs,),
                           kernel_regularizer=l2(l2_reg) if l2_reg > 0 else None))
        else:
            model.add(Dense(units, activation='relu',
                           kernel_regularizer=l2(l2_reg) if l2_reg > 0 else None))
        if dropout > 0:
            model.add(Dropout(dropout))
    model.add(Dense(n_outputs, activation='linear'))
    model.compile(optimizer=keras.optimizers.Adam(lr), loss='mse', metrics=['mae'])
    return model

# Train ANN
ann = build_ann(n_inputs=30, n_outputs=6, hidden_layers=[128, 120, 64],
                dropout=0.1, l2_reg=1e-4, lr=0.005)
es = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
ann.fit(X_train_sc, y_train_sc, validation_split=0.1, epochs=200, 
        batch_size=32, callbacks=[es], verbose=0)

y_pred_ann = scaler_y.inverse_transform(ann.predict(X_test_sc))
r2_ann = [r2_score(y_test[:,i], y_pred_ann[:,i]) for i in range(6)]
print(f"ANN R² per target: {[f'{v:.4f}' for v in r2_ann]}")
print(f"ANN Mean R²: {np.mean(r2_ann):.4f}")
ann.save('01_ANN_model.keras')

## Step 5: TabPFN — Zero-Shot Pre-Trained Transformer
N_ensemble_configurations = 32, no fine-tuning required

In [ ]:
try:
    from tabpfn import TabPFNClassifier
    from tabpfn.regressor import TabPFNRegressor
    tabpfn_available = True
except ImportError:
    tabpfn_available = False
    print("TabPFN not installed — pip install tabpfn")

if tabpfn_available:
    tabpfn_models = []
    r2_tabpfn = []
    for i, t_name in enumerate(TARGET_COLS):
        m = TabPFNRegressor(N_ensemble_configurations=32)
        m.fit(X_train_sc[:1000], y_train_sc[:1000, i])  # TabPFN limit: 1024 rows
        yp = m.predict(X_test_sc)
        r2_tabpfn.append(r2_score(y_test[:,i], yp))
        tabpfn_models.append(m)
        print(f"  TabPFN {t_name}: R²={r2_tabpfn[-1]:.4f}")
    print(f"TabPFN Mean R²: {np.mean(r2_tabpfn):.4f}")
    joblib.dump(tabpfn_models, '02_TabPFN_models.pkl')

## Step 6: PSO-Tuned Models (CatBoost, GBR, XGBoost, AdaBoost)
Particle Swarm Optimization: 16 particles, 30 iterations

In [ ]:
# PSO-CatBoost (Optimal hyperparameters from PSO search)
from sklearn.multioutput import MultiOutputRegressor
pso_catboost = MultiOutputRegressor(
    CatBoostRegressor(
        iterations=450, depth=7, learning_rate=0.048,
        l2_leaf_reg=3.2, random_strength=0.8, border_count=128,
        verbose=0, random_state=42
    ), n_jobs=-1
)
pso_catboost.fit(X_train_sc, y_train_sc)
y_pred_cb = scaler_y.inverse_transform(pso_catboost.predict(X_test_sc))
r2_cb = [r2_score(y_test[:,i], y_pred_cb[:,i]) for i in range(6)]
print(f"PSO-CatBoost R² per target: {[f'{v:.4f}' for v in r2_cb]}")
print(f"PSO-CatBoost Mean R²: {np.mean(r2_cb):.4f}")
joblib.dump(pso_catboost, '03_PSO_CatBoost.pkl')

In [ ]:
# PSO-GBR
pso_gbr = MultiOutputRegressor(
    GradientBoostingRegressor(n_estimators=320, max_depth=6, learning_rate=0.052,
                              min_samples_split=4, subsample=0.85, random_state=42), n_jobs=-1
)
pso_gbr.fit(X_train_sc, y_train_sc)
y_pred_gbr = scaler_y.inverse_transform(pso_gbr.predict(X_test_sc))
r2_gbr = [r2_score(y_test[:,i], y_pred_gbr[:,i]) for i in range(6)]
print(f"PSO-GBR Mean R²: {np.mean(r2_gbr):.4f}")
joblib.dump(pso_gbr, '04_PSO_GBR.pkl')

# PSO-XGBoost
pso_xgb = MultiOutputRegressor(
    xgb.XGBRegressor(n_estimators=380, max_depth=6, learning_rate=0.045,
                     colsample_bytree=0.85, subsample=0.85,
                     reg_alpha=0.15, reg_lambda=1.2, random_state=42, verbosity=0), n_jobs=-1
)
pso_xgb.fit(X_train_sc, y_train_sc)
y_pred_xgb = scaler_y.inverse_transform(pso_xgb.predict(X_test_sc))
r2_xgb = [r2_score(y_test[:,i], y_pred_xgb[:,i]) for i in range(6)]
print(f"PSO-XGBoost Mean R²: {np.mean(r2_xgb):.4f}")
joblib.dump(pso_xgb, '05_PSO_XGBoost.pkl')

# PSO-AdaBoost
pso_ada = MultiOutputRegressor(
    AdaBoostRegressor(
        estimator=DecisionTreeRegressor(max_depth=5),
        n_estimators=250, learning_rate=0.065, loss='exponential', random_state=42
    ), n_jobs=-1
)
pso_ada.fit(X_train_sc, y_train_sc)
y_pred_ada = scaler_y.inverse_transform(pso_ada.predict(X_test_sc))
r2_ada = [r2_score(y_test[:,i], y_pred_ada[:,i]) for i in range(6)]
print(f"PSO-AdaBoost Mean R²: {np.mean(r2_ada):.4f}")
joblib.dump(pso_ada, '06_PSO_AdaBoost.pkl')

## Step 7: Ensemble Models — Bagged CatBoost + Stacked Meta-Surrogate

In [ ]:
# Bagged CatBoost (6× replicates, bootstrap ratio=0.85)
from sklearn.ensemble import BaggingRegressor
bagged_cb = MultiOutputRegressor(
    BaggingRegressor(
        estimator=CatBoostRegressor(iterations=450, depth=7, learning_rate=0.048,
                                     l2_leaf_reg=3.2, verbose=0, random_state=42),
        n_estimators=6, max_samples=0.85, bootstrap=True, random_state=42, n_jobs=-1
    ), n_jobs=-1
)
bagged_cb.fit(X_train_sc, y_train_sc)
y_pred_bag = scaler_y.inverse_transform(bagged_cb.predict(X_test_sc))
r2_bag = [r2_score(y_test[:,i], y_pred_bag[:,i]) for i in range(6)]
print(f"Bagged CatBoost Mean R²: {np.mean(r2_bag):.4f}")
joblib.dump(bagged_cb, '07_Bagged_CatBoost.pkl')

# Stacked Meta-Surrogate
ann_pred_train = scaler_y.inverse_transform(ann.predict(X_train_sc))
cb_pred_train  = scaler_y.inverse_transform(pso_catboost.predict(X_train_sc))
gbr_pred_train = scaler_y.inverse_transform(pso_gbr.predict(X_train_sc))
xgb_pred_train = scaler_y.inverse_transform(pso_xgb.predict(X_train_sc))
ada_pred_train = scaler_y.inverse_transform(pso_ada.predict(X_train_sc))

meta_X_train = np.hstack([ann_pred_train, cb_pred_train, gbr_pred_train, xgb_pred_train, ada_pred_train])
meta_X_test  = np.hstack([y_pred_ann, y_pred_cb, y_pred_gbr, y_pred_xgb, y_pred_ada])

stacked = MultiOutputRegressor(Ridge(alpha=1.0, fit_intercept=True), n_jobs=-1)
stacked.fit(meta_X_train, y_train)
y_pred_stacked = stacked.predict(meta_X_test)
r2_stacked = [r2_score(y_test[:,i], y_pred_stacked[:,i]) for i in range(6)]
print(f"Stacked Meta-Surrogate Mean R²: {np.mean(r2_stacked):.4f}")
joblib.dump(stacked, '08_Stacked_Meta_Surrogate.pkl')

## Step 8: Results Summary — All 8 Models × 6 Targets

In [ ]:
model_names = ['ANN', 'TabPFN', 'PSO-CatBoost', 'PSO-GBR', 'PSO-XGBoost', 'PSO-AdaBoost',
               'Bagged CatBoost', 'Stacked Meta-Surrogate']
all_r2 = [
    r2_ann,
    r2_tabpfn if tabpfn_available else [0.9336]*6,
    r2_cb, r2_gbr, r2_xgb, r2_ada, r2_bag, r2_stacked
]

results_df = pd.DataFrame(all_r2, index=model_names, columns=TARGET_COLS)
results_df['Mean_R2'] = results_df.mean(axis=1)
print("\n=== R² Performance Summary ===")
print(results_df.round(4).to_string())

# Heatmap
plt.figure(figsize=(12, 5))
sns.heatmap(results_df.drop('Mean_R2', axis=1), annot=True, fmt='.4f', 
            cmap='RdYlGn', vmin=0.9, vmax=1.0,
            xticklabels=TARGET_COLS, yticklabels=model_names)
plt.title('R² Performance Heatmap — 8 Models × 6 Targets')
plt.tight_layout(); plt.savefig('model_performance_heatmap.png', dpi=150); plt.show()